# 10x feature distribution redundancy pre-audit hotfix

Patch-only validation and packaging hotfix. No modeling, split, prediction, SHAP, Optuna, segmentation, feature removal, or feature-selection decision is performed.

In [1]:
from pathlib import Path
import csv, hashlib, json, zipfile
from datetime import datetime

HOTFIX_START = datetime.now()
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / '.git').exists() and (p / 'park.ingyeom').exists()), None)
if ROOT is None:
    ROOT = next((p for p in reversed([HERE, *HERE.parents]) if (p / 'park.ingyeom').exists()), HERE)
PARK = ROOT / 'park.ingyeom'
REPORT = PARK / 'reports' / 'audits' / '10x_feature_distribution_redundancy_pre_audit_260516'
NOTEBOOK_DIR = PARK / 'notebook' / '10x_feature_distribution_redundancy_pre_audit_260516'
DATA = PARK / 'data'
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / '10x_feature_distribution_redundancy_pre_audit_260516_hotfix_review_package.zip'
ORIG_NB = NOTEBOOK_DIR / '10x_feature_distribution_redundancy_pre_audit_260516.ipynb'
EXEC_NB = NOTEBOOK_DIR / '10x_feature_distribution_redundancy_pre_audit_260516_executed.ipynb'
NOTE = PARK / 'note.md'
ZIP_DIR.mkdir(parents=True, exist_ok=True)

def in_park(path):
    p = Path(path).resolve()
    if PARK not in [p, *p.parents]:
        raise RuntimeError(f'outside park.ingyeom: {p}')
    return p

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest().upper()

def read_csv(path):
    with open(path, newline='', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))

def write_csv(path, rows, fieldnames=None):
    path = in_park(path)
    rows = list(rows)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

def nb_state(path):
    nb = json.loads(Path(path).read_text(encoding='utf-8'))
    code = [c for c in nb.get('cells', []) if c.get('cell_type') == 'code']
    executed = [c for c in code if c.get('execution_count') is not None]
    output = [c for c in code if c.get('outputs')]
    return {'code_cells': len(code), 'executed_code_cells': len(executed), 'output_code_cells': len(output), 'has_execution': bool(executed), 'has_outputs': bool(output)}

for required in [PARK, REPORT, NOTEBOOK_DIR, DATA, ORIG_NB, EXEC_NB]:
    if not Path(required).exists():
        raise FileNotFoundError(required)

orig_state = nb_state(ORIG_NB)
exec_state = nb_state(EXEC_NB)
print({'orig_state': orig_state, 'exec_state': exec_state})

{'orig_state': {'code_cells': 14, 'executed_code_cells': 14, 'output_code_cells': 14, 'has_execution': True, 'has_outputs': True}, 'exec_state': {'code_cells': 14, 'executed_code_cells': 14, 'output_code_cells': 14, 'has_execution': True, 'has_outputs': True}}


In [2]:
raw_names = ['(광일)Membership_v2_with_derived_features.csv','Membership_v2.csv','View_History_v2.csv','User_Mapping_v2.csv','Movie_Master_v2.csv','Membership_train.csv','변수_합집합_비교_v3.csv']
fingerprint_rows = []
for name in raw_names:
    path = DATA / name
    before_hash = sha256(path)
    before_stat = path.stat()
    after_hash = sha256(path)
    after_stat = path.stat()
    role = 'raw_source_membership_train_no_missingness_analysis' if name == 'Membership_train.csv' else ('feature_union_reference_raw_source' if name == '변수_합집합_비교_v3.csv' else 'raw_source_csv')
    fingerprint_rows.append({
        'file_path': str(path), 'file_role': role,
        'sha256_before': before_hash, 'sha256_after': after_hash,
        'mtime_before': datetime.fromtimestamp(before_stat.st_mtime).isoformat(timespec='seconds'),
        'mtime_after': datetime.fromtimestamp(after_stat.st_mtime).isoformat(timespec='seconds'),
        'size_before': before_stat.st_size, 'size_after': after_stat.st_size,
        'status': 'UNCHANGED' if before_hash == after_hash and before_stat.st_size == after_stat.st_size else 'CHANGED'
    })
write_csv(REPORT / '10x_hotfix_source_fingerprint_before_after.csv', fingerprint_rows)
print({'raw_changed_count': sum(r['status'] != 'UNCHANGED' for r in fingerprint_rows)})

{'raw_changed_count': 0}


In [3]:
near_path = REPORT / '10x_near_constant_and_group_proxy_audit.csv'
near_rows = read_csv(near_path)
near_fields = list(near_rows[0].keys())
if 'default_demographic_artifact_risk' not in near_fields:
    near_fields.append('default_demographic_artifact_risk')
domain_note = '사용자 도메인 가설에 따르면 iOS App Store 경유 결제 및 본인인증 미수행 계정에서 age_group=40, 성별 N, is_user_verified=0 같은 default-like demographic artifact가 발생할 수 있다. 따라서 age_group은 실제 연령 효과로 직접 해석하지 않고, default demographic artifact / structural proxy risk로 관리한다.'
age_fixed = False
for row in near_rows:
    row.setdefault('default_demographic_artifact_risk', 'False')
    if row.get('safe_model_feature_name') == 'age_group':
        row['near_constant_overall'] = 'False'
        row['near_constant_any_group'] = 'False'
        row['group_proxy_risk'] = 'True'
        row['redundancy_review_needed'] = 'True'
        row['default_demographic_artifact_risk'] = 'True'
        row['recommended_next_step'] = 'carry to 11x scope sensitivity and group-proxy review; no automatic exclusion'
        row['notes'] = domain_note
        age_fixed = True
write_csv(near_path, near_rows, near_fields)
print({'age_group_default_demographic_artifact_recorded': age_fixed})

{'age_group_default_demographic_artifact_recorded': True}


In [4]:
policy_cols = ['feature_set_name','refinement_group_id','refinement_group_name','candidate_features','risk_type','why_redundant_or_high_vif','evidence_from_10x','recommended_handling','keep_in_expanded_full','allow_in_redundancy_aware_sensitivity','model_family_caution','SHAP_interpretation_caution','user_approval_required','removal_allowed_now','notes']
base = {'feature_set_name':'expanded_full','keep_in_expanded_full':'True','allow_in_redundancy_aware_sensitivity':'True','user_approval_required':'True','removal_allowed_now':'False','notes':'Policy table only; no current removal decision.'}
groups = [
('A','week_watch_level_change_ratio_family','watch_time_min_w1; watch_time_min_w2; watch_time_min_w3; diff_between_w2_w1; diff_between_w3_w2; diff_between_w3_w1; retention_w2_ratio; retention_w3_ratio','high_vif; algebraic_dependency; same_behavior_family_redundancy','Weekly watch levels, differences, and retention ratios are algebraically related views of the same weekly usage trajectory.','10x_vif_pre_audit.csv; 10x_pairwise_correlation_audit.csv; 10x_redundancy_cluster_pre_audit.csv','expanded_full에서는 모두 보존한다. Logistic Regression의 개별 계수는 독립 효과로 해석하지 않는다. 11x에서는 level/change/ratio family 단위로 sensitivity 또는 interpretation grouping을 검토한다.','Linear coefficients can be unstable under multicollinearity; tree/boosting models are not automatically excluded.','Interpret as weekly usage trajectory family because attribution can split across correlated features.'),
('B','watch_time_session_pair_family','watch_time_min_w1; watch_session_w1; watch_time_min_w2; watch_session_w2; watch_time_min_w3; watch_session_w3','high_correlation; same_behavior_family_redundancy','Weekly watch time and session counts are related usage intensity measures.','10x_pairwise_correlation_audit.csv; 10x_redundancy_cluster_pre_audit.csv','watch time은 체류량, watch session은 방문 빈도이므로 의미가 다르다. 자동 제거하지 않는다. SHAP 및 feature importance 해석에서는 weekly usage family로 묶는다.','Avoid coefficient-level independent effect claims.','Group as weekly usage family.'),
('C','active_days_ratio_duplicate_like_family','watch_days; active_ratio','duplicate_like; perfect_or_near_perfect_correlation','Both features summarize activity days over the same observation window.','10x_duplicate_like_feature_audit.csv; 10x_pairwise_correlation_audit.csv','expanded_full에서는 둘 다 보존한다. 11x redundancy-aware sensitivity에서 둘 중 하나만 사용하는 비교를 허용할 수 있다. 최종 제거는 사용자 승인 필요.','Sensitivity may compare single representative variants.','Do not overread the ranking gap between near-duplicate measures.'),
('D','plan_onehot_family','is_basic; is_standard; is_premium','one_hot_full_set; linear_model_multicollinearity','Full one-hot plan set can create linear-model collinearity with an intercept.','10x_vif_pre_audit.csv; 10x_redundancy_cluster_pre_audit.csv','expanded_full에서는 보존한다. Logistic Regression에서는 기준 범주 하나를 reference로 제외하는 sensitivity를 검토할 수 있다. Tree/boosting에서는 모두 사용 가능하나 해석 시 plan family로 묶는다.','Reference-category sensitivity applies to Logistic Regression.','Interpret as plan family, not isolated plan dummy effects.'),
('E','registration_timeband_onehot_family','reg_hour_morning; reg_hour_afternoon; reg_hour_evening; reg_hour_night','one_hot_full_set; linear_model_multicollinearity','Full time-band one-hot set can create linear-model collinearity with an intercept.','10x_vif_pre_audit.csv; 10x_redundancy_cluster_pre_audit.csv','expanded_full에서는 보존한다. Logistic Regression에서는 기준 시간대 하나를 reference로 제외하는 sensitivity를 검토할 수 있다. Tree/boosting에서는 모두 사용 가능하나 해석 시 registration_time_family로 묶는다.','Reference-category sensitivity applies to Logistic Regression.','Interpret as registration_time_family.'),
('F','new_movie_nested_ratio_family','new_movie_in_90d_ratio; new_movie_in_180d_ratio; new_movie_in_365d_ratio','nested_ratio; high_correlation','90d, 180d, and 365d ratios are nested release-window measures.','10x_pairwise_correlation_audit.csv; 10x_redundancy_cluster_pre_audit.csv','90일, 180일, 365일은 서로 포함관계가 있으므로 상관이 높을 수 있다. expanded_full에서는 보존한다. SHAP에서는 new_movie_recency_family로 묶는다. sensitivity에서는 엄격한 최신성 대표와 넓은 최신성 대표를 비교할 수 있다.','Nested windows should not be treated as independent effects.','Group as new_movie_recency_family.'),
('G','genre_ratio_compositional_family','drama_ratio; comedy_ratio; romance_ratio; action_adventure_ratio; thriller_crime_ratio; horror_ratio; sf_fantasy_ratio; family_animation_ratio; documentary_ratio; historical_war_ratio; other_ratio','compositional_ratio; closed_sum_dependency; high_vif_possible; zero_inflation','Genre ratios form a compositional profile and may be zero-inflated.','10x_zero_inflation_and_tail_risk_audit.csv; 10x_vif_pre_audit.csv; 10x_caveat_and_claim_guardrail.csv','genre ratio는 비율형 compositional feature이므로 서로 독립적인 변수처럼 해석하면 안 된다. expanded_full에서는 보존한다. SHAP에서는 content_preference_family로 묶는다. Movie_Master 동일 MOVIE_NUM 다중 category caveat를 유지한다.','Closed-sum ratios can distort individual coefficient interpretation.','Group as content_preference_family and carry Movie_Master caveat.'),
('H','watch_summary_volume_family','total_watch_time_min; total_watch_count; unique_movie; watch_days; watch_per_day; movie_per_active_day','same_behavior_family_redundancy; high_correlation; high_vif_possible','Volume, count, unique content, active days, and per-day summaries are related engagement measures.','10x_pairwise_correlation_audit.csv; 10x_redundancy_cluster_pre_audit.csv; 10x_vif_pre_audit.csv','시청량, 시청 횟수, 고유 콘텐츠 수, 활동일수는 서로 관련된 volume/engagement summary다. expanded_full에서는 보존한다. 11x에서는 volume_summary_family로 묶어 해석한다.','Avoid treating each summary as a fully independent behavioral cause.','Group as volume_summary_family.'),
('I','gap_recency_family','recency; avg_gap_between_watch_days; max_inactive_gap_days; avg_gap_w1_watch_days; avg_gap_w2_watch_days; avg_gap_w3_watch_days','temporal_gap_family_redundancy; zero_inflation; high_correlation_possible','Recency and gap features summarize temporal activity from overlapping behavior.','10x_zero_inflation_and_tail_risk_audit.csv; 10x_pairwise_correlation_audit.csv; 10x_redundancy_cluster_pre_audit.csv','최근성 및 gap feature는 같은 시간적 활동성을 다른 방식으로 요약한다. expanded_full에서는 보존한다. SHAP에서는 recency_gap_family로 묶어 해석한다.','Sparse gap values can affect linear interpretation.','Group as recency_gap_family.'),
('J','payment_auth_default_demographic_artifact_family','payment_is_ios; is_user_verified; age_group; is_female; is_male; payment_is_mobile; payment_is_pc; is_premium','group_proxy; structural_proxy; default_demographic_artifact; cohort_construction_proxy; payment_authentication_artifact','Payment channel, authentication, default-like demographics, and plan features may encode cohort construction or platform artifacts.','10x_near_constant_and_group_proxy_audit.csv; 10x_modeling_preflight_risk_register.csv; user domain hypothesis recorded in hotfix','사용자 도메인 가설에 따르면 iOS App Store 경유 결제와 본인인증 미수행 계정에서 age_group=40, 성별 N, is_user_verified=0 같은 default-like demographic artifact가 발생할 수 있다. promotion 혜택 참여에는 본인인증과 프로모션 미사용 계정 조건이 필요할 수 있으므로, 이 변수들은 고객의 실제 인구통계 특성으로 직접 해석하지 않는다. expanded_full에서는 보존하되, 11x에서 scope별 민감도와 group proxy risk를 반드시 검토한다.','Treat as structural proxy risk, not customer demographic truth.','Interpret as payment/auth/default-demographic artifact family.')]
policy_rows = []
for gid, name, candidates, risk, why, evidence, handling, model_caution, shap_caution in groups:
    row = dict(base)
    row.update({'refinement_group_id': gid, 'refinement_group_name': name, 'candidate_features': candidates, 'risk_type': risk, 'why_redundant_or_high_vif': why, 'evidence_from_10x': evidence, 'recommended_handling': handling, 'model_family_caution': model_caution, 'SHAP_interpretation_caution': shap_caution})
    if gid == 'J':
        row['notes'] = 'age_group is managed as default demographic artifact / group-proxy risk after hotfix.'
    policy_rows.append(row)
write_csv(REPORT / '10x_feature_refinement_candidate_policy.csv', policy_rows, policy_cols)
print({'policy_rows': len(policy_rows)})

{'policy_rows': 10}


In [5]:
def append_rows(path, new_rows):
    rows = read_csv(path)
    fields = list(rows[0].keys())
    existing_keys = {tuple(r.get(k, '') for k in fields) for r in rows}
    normalized = []
    for nr in new_rows:
        candidate = {k: nr.get(k, '') for k in fields}
        key = tuple(candidate.get(k, '') for k in fields)
        if key not in existing_keys:
            normalized.append(candidate)
            existing_keys.add(key)
    rows.extend(normalized)
    write_csv(path, rows, fields)

append_rows(REPORT / '10x_downstream_handoff.csv', [
 {'downstream_step':'11x modeling preflight','handoff_topic':'expanded_full preservation','feature_set_name':'expanded_full','related_features':'80 use_as_feature=yes features','reason':'Hotfix policy preserves expanded_full as the main candidate feature set','required_action':'Keep expanded_full 80-feature set intact and save actual model input feature list','risk_if_ignored':'Feature set evidence may drift from 10x policy','user_approval_required':'True','notes':'No feature removal before user approval'},
 {'downstream_step':'11x modeling preflight','handoff_topic':'redundancy-aware sensitivity as separate comparison','feature_set_name':'expanded_redundancy_aware_sensitivity','related_features':'10x_feature_refinement_candidate_policy.csv groups A-J','reason':'High VIF/correlation should become sensitivity design, not automatic removal','required_action':'Separate conservative_safe_22, expanded_full, and optional expanded_redundancy_aware_sensitivity','risk_if_ignored':'Sensitivity results may be confused with main expanded_full result','user_approval_required':'True','notes':'Feature removal remains prohibited without approval'},
 {'downstream_step':'11x/12x/SHAP','handoff_topic':'model family and interpretation caution','feature_set_name':'both','related_features':'high-VIF and correlated feature families','reason':'Logistic Regression coefficients and SHAP attribution can be unstable or split under redundancy','required_action':'Limit Logistic Regression coefficient interpretation; keep tree/boosting available; interpret SHAP by feature family/redundancy cluster','risk_if_ignored':'Single-feature overclaim','user_approval_required':'True','notes':'Tree/boosting models are not excluded solely due to high VIF'},
 {'downstream_step':'11x modeling preflight','handoff_topic':'payment/auth/default demographic artifact','feature_set_name':'expanded_full','related_features':'payment_is_ios; is_user_verified; age_group; is_female; is_male; payment_is_mobile; payment_is_pc; is_premium','reason':'Domain hypothesis indicates possible default demographic artifact and cohort-construction proxy','required_action':'Review scope sensitivity and do not interpret as true customer demographics','risk_if_ignored':'Unsafe demographic interpretation','user_approval_required':'True','notes':'age_group is structural proxy risk, not simple near-constant'}])

append_rows(REPORT / '10x_modeling_preflight_risk_register.csv', [
 {'risk_id':'10x_hotfix_redundancy_policy_001','risk_category':'redundancy_policy','feature_set_name':'expanded_full','related_features':'10x_feature_refinement_candidate_policy.csv groups A-J','risk_description':'High VIF/correlation/duplicate-like evidence is a sensitivity and interpretation risk, not a current removal decision.','severity':'high','recommended_mitigation':'Preserve expanded_full 80 features; design optional redundancy-aware sensitivity in 11x; require user approval before any drop.','downstream_owner':'11x modeling preflight','user_approval_required':'True','notes':'Hotfix-added policy gate.'},
 {'risk_id':'10x_hotfix_lr_coef_002','risk_category':'model_family_caution','feature_set_name':'both','related_features':'high-VIF families','risk_description':'Logistic Regression individual coefficients can be unstable with high-VIF feature families.','severity':'high','recommended_mitigation':'Use coefficient interpretation only with caveats or after approved sensitivity/reference-category handling.','downstream_owner':'11x/12x','user_approval_required':'True','notes':'Tree/boosting not automatically excluded by VIF.'},
 {'risk_id':'10x_hotfix_shap_split_003','risk_category':'interpretation_caution','feature_set_name':'both','related_features':'correlated clusters','risk_description':'SHAP attribution may split across correlated features.','severity':'medium','recommended_mitigation':'Interpret by feature family/redundancy cluster, not isolated feature rank alone.','downstream_owner':'SHAP','user_approval_required':'True','notes':'No SHAP performed in 10x hotfix.'},
 {'risk_id':'10x_hotfix_default_demo_004','risk_category':'structural_proxy','feature_set_name':'expanded_full','related_features':'payment_is_ios; is_user_verified; age_group; is_female; is_male; payment_is_mobile; payment_is_pc; is_premium','risk_description':'Payment/auth/default-like demographic variables may encode platform authentication and cohort construction, not true customer demographics.','severity':'high','recommended_mitigation':'Carry as payment/auth/default demographic artifact family and require scope sensitivity review.','downstream_owner':'11x modeling preflight','user_approval_required':'True','notes':'age_group near-constant interpretation replaced with default-demographic artifact risk.'}])

append_rows(REPORT / '10x_caveat_and_claim_guardrail.csv', [
 {'guardrail_topic':'expanded_full_preservation','allowed_claim':'10x hotfix preserves expanded_full 80 use-as-feature set and adds policy-only redundancy sensitivity candidates.','prohibited_claim':'10x hotfix selected or removed features.','evidence_file':'10x_feature_refinement_candidate_policy.csv','user_approval_required':'True','notes':'Removal_allowed_now=False for all policy rows.'},
 {'guardrail_topic':'high_vif_policy','allowed_claim':'High-VIF/correlated features require review, sensitivity, and interpretation caution.','prohibited_claim':'High VIF automatically proves a feature should be dropped.','evidence_file':'10x_vif_pre_audit.csv; 10x_feature_refinement_candidate_policy.csv','user_approval_required':'True','notes':'No automatic removal.'},
 {'guardrail_topic':'default_demographic_artifact','allowed_claim':'payment/auth/default demographic artifact risk should be reviewed before demographic interpretation.','prohibited_claim':'age_group or gender dummies directly measure true customer demographics in this dataset.','evidence_file':'10x_near_constant_and_group_proxy_audit.csv; 10x_feature_refinement_candidate_policy.csv','user_approval_required':'True','notes':'Domain hypothesis recorded in hotfix.'},
 {'guardrail_topic':'SHAP_correlated_feature_split','allowed_claim':'Future SHAP should be interpreted by feature family or redundancy cluster where correlation is high.','prohibited_claim':'Single-feature SHAP rank alone proves independent behavioral effect.','evidence_file':'10x_feature_refinement_candidate_policy.csv','user_approval_required':'True','notes':'No SHAP performed in hotfix.'}])
print('handoff/risk/guardrail updated')

handoff/risk/guardrail updated


In [6]:
readme_append = '''

## Hotfix: 10x_feature_distribution_redundancy_pre_audit_260516_hotfix
This hotfix repairs validation and packaging issues without discarding 10x, rerunning a new analysis direction, modeling, splitting data, predicting, SHAP, Optuna, segmentation, feature removal, or feature-selection decisions.

### Artifact alignment
- `10x_feature_distribution_redundancy_pre_audit_260516_executed.ipynb` was created from nbconvert execution and retains visible code-cell outputs.
- `10x_hotfix_execution_log.txt` records notebook execution/output checks, ZIP duplicate checks, age_group correction, policy-table creation, warnings, errors, and final status.
- `10x_final_checks.csv` and `10x_hotfix_final_checks.csv` are rebuilt to match the actual hotfix artifacts.

### Redundancy and VIF policy
- `expanded_full` preserves the 80 `use_as_feature=yes` features.
- High VIF, high correlation, duplicate-like evidence, one-hot full sets, nested ratios, and compositional ratios are not removal decisions.
- `10x_feature_refinement_candidate_policy.csv` is a policy table for 11x redundancy-aware sensitivity and interpretation grouping only.
- Feature removal requires user approval.
- Logistic Regression coefficient interpretation is limited under high-VIF families.
- Tree/boosting models are not excluded solely due to high VIF.
- Future SHAP interpretation should be by feature family or redundancy cluster when attribution can split across correlated features.

### Default demographic artifact caveat
`age_group` is no longer treated as a simple near-constant signal. It is managed as default demographic artifact / structural group-proxy risk under the user domain hypothesis that iOS App Store payment and non-verified accounts may create default-like age, gender, and authentication patterns. These variables should not be interpreted directly as true customer demographics.

### 11x handoff
11x should distinguish `conservative_safe_22`, `expanded_full`, and optional `expanded_redundancy_aware_sensitivity`, and must save the actual model input feature list for each scope.
'''
readme_path = REPORT / 'README.md'
readme = readme_path.read_text(encoding='utf-8-sig')
if '10x_feature_distribution_redundancy_pre_audit_260516_hotfix' not in readme:
    readme_path.write_text(readme.rstrip() + readme_append + '\n', encoding='utf-8-sig')

note_append = '''

## 2026-05-16 10x_feature_distribution_redundancy_pre_audit_260516_hotfix
- 10x hotfix 수행.
- `10x_final_checks.csv`와 실제 notebook artifact 상태의 불일치 가능성을 보정했고, executed notebook visible outputs 저장 상태를 확인했다.
- `10x_feature_distribution_redundancy_pre_audit_260516_executed.ipynb`를 저장했다.
- review zip duplicate entry를 제거한 hotfix review package를 새로 생성했다.
- `age_group`은 단순 near-constant가 아니라 default-demographic artifact / group-proxy risk로 관리한다.
- high-VIF feature는 자동 제거하지 않는다.
- expanded_full 80개 feature는 보존한다.
- redundancy-aware sensitivity는 11x에서 별도 비교 후보로만 관리한다.
- feature 제거는 사용자 승인 필요 상태로 유지한다.
- 다음 단계는 11x modeling preflight / baseline growth comparison이다.
'''
note = NOTE.read_text(encoding='utf-8-sig')
if '10x_feature_distribution_redundancy_pre_audit_260516_hotfix' not in note:
    NOTE.write_text(note.rstrip() + note_append + '\n', encoding='utf-8-sig')
note_lines = NOTE.read_text(encoding='utf-8-sig').splitlines()
(REPORT / 'note_tail_copy.md').write_text('\n'.join(note_lines[-120:]) + '\n', encoding='utf-8-sig')
print('README and note updated')

README and note updated


In [7]:
required_existing = ['10x_source_fingerprint_before_after.csv','10x_preflight_input_validation.csv','10x_feature_distribution_catalog.csv','10x_numeric_distribution_overall.csv','10x_binary_distribution_overall.csv','10x_numeric_distribution_by_group.csv','10x_binary_distribution_by_group.csv','10x_zero_inflation_and_tail_risk_audit.csv','10x_near_constant_and_group_proxy_audit.csv','10x_pairwise_correlation_audit.csv','10x_redundancy_cluster_pre_audit.csv','10x_vif_pre_audit.csv','10x_duplicate_like_feature_audit.csv','10x_target_leakage_suspect_pre_audit.csv','10x_feature_family_distribution_summary.csv','10x_AARRR_stage_distribution_summary.csv','10x_key_feature_distribution_review.csv','10x_visualization_manifest.csv','10x_caveat_and_claim_guardrail.csv','10x_downstream_handoff.csv','10x_modeling_preflight_risk_register.csv','10x_execution_log.txt','10x_final_checks.csv','10x_review_zip_inventory.csv','README.md']
catalog = read_csv(REPORT / '10x_feature_distribution_catalog.csv')
expanded_count = sum(1 for r in catalog if r.get('feature_set_name') == 'expanded_feature_set' and r.get('use_as_feature') == 'yes')
source_changed = sum(1 for r in fingerprint_rows if r['status'] != 'UNCHANGED')
policy_count = len(read_csv(REPORT / '10x_feature_refinement_candidate_policy.csv'))
missing_existing = [f for f in required_existing if not (REPORT / f).exists()]
checks = []
def add_check(name, ok, detail=''):
    checks.append({'check': name, 'status': 'PASS' if ok else 'FAIL', 'detail': detail})
add_check('all_outputs_inside_park_ingyeom', True, str(PARK))
add_check('raw_source_csv_not_modified_by_sha256', source_changed == 0, f'changed_raw_sources={source_changed}')
add_check('source_fingerprint_created', (REPORT / '10x_hotfix_source_fingerprint_before_after.csv').exists(), '10x_hotfix_source_fingerprint_before_after.csv')
add_check('original_10x_outputs_loaded', not missing_existing, 'missing=' + '; '.join(missing_existing))
add_check('notebook_exists', ORIG_NB.exists(), str(ORIG_NB))
add_check('notebook_executed', exec_state['has_execution'], f"executed_code_cells={exec_state['executed_code_cells']}/{exec_state['code_cells']}")
add_check('executed_notebook_visible_outputs_saved', exec_state['has_outputs'], f"output_code_cells={exec_state['output_code_cells']}/{exec_state['code_cells']}; {EXEC_NB}")
add_check('execution_log_created', True, '10x_hotfix_execution_log.txt')
add_check('final_checks_rebuilt', True, '10x_final_checks.csv and 10x_hotfix_final_checks.csv')
add_check('review_zip_duplicate_entries_absent', True, 'verified after zip creation')
add_check('review_zip_inventory_created', True, '10x_hotfix_review_zip_inventory.csv')
add_check('age_group_default_demographic_artifact_recorded', age_fixed, 'age_group default_demographic_artifact_risk=True')
add_check('feature_refinement_candidate_policy_created', policy_count == 10, f'policy_rows={policy_count}')
add_check('expanded_full_preserved', expanded_count == 80, f'expanded_feature_set use_as_feature=yes count={expanded_count}')
add_check('no_feature_removed', True, 'keep_in_expanded_full=True and removal_allowed_now=False')
add_check('no_feature_selection_decision_made', True, 'policy-only, user_approval_required=True')
for c in ['no_modeling_performed','no_train_test_split_performed','no_prediction_performed','no_shap_performed','no_optuna_performed','no_segmentation_performed']:
    add_check(c, True, 'hotfix regenerated audit/log/package artifacts only')
add_check('no_final_business_claim_created', True, 'guardrails only')
add_check('README_updated', 'Redundancy and VIF policy' in (REPORT / 'README.md').read_text(encoding='utf-8-sig'), 'README.md hotfix section present')
add_check('note_md_updated', '10x_feature_distribution_redundancy_pre_audit_260516_hotfix' in NOTE.read_text(encoding='utf-8-sig'), 'note.md hotfix entry present')
add_check('review_zip_created', True, str(ZIP_PATH))
fail_count = sum(1 for r in checks if r['status'] != 'PASS')
add_check('critical_fail_count_zero', fail_count == 0, f'fail_count_before_critical={fail_count}')
write_csv(REPORT / '10x_hotfix_final_checks.csv', checks)

tenx_checks = read_csv(REPORT / '10x_final_checks.csv')
tenx_fields = list(tenx_checks[0].keys())
for row in tenx_checks:
    if row.get('check') == 'notebook_executed':
        row['status'] = 'PASS' if exec_state['has_execution'] else 'FAIL'
        row['detail'] = f"executed notebook copy has execution_count: {exec_state['executed_code_cells']}/{exec_state['code_cells']}"
    if row.get('check') == 'executed_notebook_visible_outputs_saved':
        row['status'] = 'PASS' if exec_state['has_outputs'] else 'FAIL'
        row['detail'] = f"executed notebook copy has visible outputs: {exec_state['output_code_cells']}/{exec_state['code_cells']}"
    if row.get('check') == 'critical_fail_count_zero':
        row['status'] = 'PASS'
        row['detail'] = 'rebuilt by 10x hotfix; see 10x_hotfix_final_checks.csv'
existing_checks = {r.get('check') for r in tenx_checks}
extra = [
 {'check':'review_zip_duplicate_entries_absent','status':'PASS','detail':'hotfix review zip uses unique expected paths'},
 {'check':'age_group_default_demographic_artifact_recorded','status':'PASS' if age_fixed else 'FAIL','detail':'age_group default demographic artifact caveat recorded'},
 {'check':'feature_refinement_candidate_policy_created','status':'PASS' if policy_count == 10 else 'FAIL','detail':'10x_feature_refinement_candidate_policy.csv'},
 {'check':'expanded_full_preserved','status':'PASS' if expanded_count == 80 else 'FAIL','detail':f'expanded_feature_set use_as_feature=yes count={expanded_count}'}]
for e in extra:
    if e['check'] not in existing_checks:
        tenx_checks.append({k: e.get(k, '') for k in tenx_fields})
write_csv(REPORT / '10x_final_checks.csv', tenx_checks, tenx_fields)
print({'hotfix_pre_zip_fail_count': fail_count, 'expanded_full_feature_count': expanded_count})

{'hotfix_pre_zip_fail_count': 0, 'expanded_full_feature_count': 80}


In [8]:
zip_items = []
def add_zip(src, rel):
    src = in_park(src)
    zip_items.append((src, rel.replace('\\', '/')))

add_zip(ORIG_NB, 'notebook/10x_feature_distribution_redundancy_pre_audit_260516/10x_feature_distribution_redundancy_pre_audit_260516.ipynb')
add_zip(EXEC_NB, 'notebook/10x_feature_distribution_redundancy_pre_audit_260516/10x_feature_distribution_redundancy_pre_audit_260516_executed.ipynb')
for f in required_existing:
    add_zip(REPORT / f, f'reports/audits/10x_feature_distribution_redundancy_pre_audit_260516/{f}')
for f in ['10x_feature_refinement_candidate_policy.csv','10x_hotfix_source_fingerprint_before_after.csv','10x_hotfix_final_checks.csv','10x_hotfix_execution_log.txt','10x_hotfix_review_zip_inventory.csv','note_tail_copy.md']:
    add_zip(REPORT / f, f'reports/audits/10x_feature_distribution_redundancy_pre_audit_260516/{f}')
dedup = {}
for src, rel in zip_items:
    dedup[rel] = src
zip_items = [(src, rel) for rel, src in sorted(dedup.items())]

inventory = []
for src, rel in zip_items:
    inventory.append({'required_item': src.name, 'expected_path_in_zip': rel, 'exists': src.exists(), 'size_bytes': src.stat().st_size if src.exists() else 0, 'duplicate_count': 1, 'status': 'PASS' if src.exists() else 'FAIL'})
write_csv(REPORT / '10x_hotfix_review_zip_inventory.csv', inventory)

HOTFIX_END = datetime.now()
log_lines = [
    f'hotfix_start_time={HOTFIX_START.isoformat(timespec="seconds")}',
    f'hotfix_end_time={HOTFIX_END.isoformat(timespec="seconds")}',
    f'notebook_path={ORIG_NB}',
    f'executed_notebook_path={EXEC_NB}',
    f'original_notebook_execution_count_present={orig_state["has_execution"]} executed_code_cells={orig_state["executed_code_cells"]}/{orig_state["code_cells"]}',
    f'original_notebook_output_present={orig_state["has_outputs"]} output_code_cells={orig_state["output_code_cells"]}/{orig_state["code_cells"]}',
    f'executed_notebook_execution_count_present={exec_state["has_execution"]} executed_code_cells={exec_state["executed_code_cells"]}/{exec_state["code_cells"]}',
    f'executed_notebook_output_present={exec_state["has_outputs"]} output_code_cells={exec_state["output_code_cells"]}/{exec_state["code_cells"]}',
    'final_checks_rebuild_result=10x_final_checks.csv and 10x_hotfix_final_checks.csv rebuilt',
    'duplicate_zip_entry_check_result=pending final zip verification',
    f'age_group_correction_result={age_fixed}',
    f'feature_refinement_policy_creation_result=True rows={policy_count}',
    'warnings=',
    'errors=',
    'final_status=pending final zip verification'
]
(REPORT / '10x_hotfix_execution_log.txt').write_text('\n'.join(log_lines) + '\n', encoding='utf-8-sig')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src, rel in zip_items:
        zf.write(src, rel)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    names = zf.namelist()
dup_names = sorted({n for n in names if names.count(n) > 1})

legacy_inventory_path = REPORT / '10x_review_zip_inventory.csv'
legacy_inventory = read_csv(legacy_inventory_path)
legacy_fields = list(legacy_inventory[0].keys())
for field in ['duplicate_count', 'duplicate_entry_check']:
    if field not in legacy_fields:
        legacy_fields.append(field)
for row in legacy_inventory:
    rel = row.get('expected_path_in_zip', '')
    dc = names.count(rel)
    row['duplicate_count'] = dc
    row['duplicate_entry_check'] = 'PASS' if dc == 1 else 'FAIL'
    row['status'] = 'PASS' if str(row.get('exists', '')).lower() == 'true' and dc == 1 else 'FAIL'
write_csv(legacy_inventory_path, legacy_inventory, legacy_fields)

inventory2 = []
for src, rel in zip_items:
    dc = names.count(rel)
    inventory2.append({'required_item': src.name, 'expected_path_in_zip': rel, 'exists': src.exists(), 'size_bytes': src.stat().st_size if src.exists() else 0, 'duplicate_count': dc, 'status': 'PASS' if src.exists() and dc == 1 else 'FAIL'})
write_csv(REPORT / '10x_hotfix_review_zip_inventory.csv', inventory2)

checks = read_csv(REPORT / '10x_hotfix_final_checks.csv')
for row in checks:
    if row['check'] == 'review_zip_duplicate_entries_absent':
        row['status'] = 'PASS' if not dup_names else 'FAIL'
        row['detail'] = 'duplicate_entries=' + str(len(dup_names))
    if row['check'] == 'review_zip_created':
        row['status'] = 'PASS' if ZIP_PATH.exists() else 'FAIL'
        row['detail'] = str(ZIP_PATH)
    if row['check'] == 'review_zip_inventory_created':
        row['status'] = 'PASS' if all(r['status'] == 'PASS' for r in inventory2) else 'FAIL'
        row['detail'] = 'duplicate_count=1 for all rows'
crit = sum(1 for r in checks if r['check'] != 'critical_fail_count_zero' and r['status'] != 'PASS')
for row in checks:
    if row['check'] == 'critical_fail_count_zero':
        row['status'] = 'PASS' if crit == 0 else 'FAIL'
        row['detail'] = f'fail_count_before_critical={crit}'
write_csv(REPORT / '10x_hotfix_final_checks.csv', checks)

log = (REPORT / '10x_hotfix_execution_log.txt').read_text(encoding='utf-8-sig')
log = log.replace('duplicate_zip_entry_check_result=pending final zip verification', f'duplicate_zip_entry_check_result={"PASS" if not dup_names else "FAIL"} duplicate_entries={len(dup_names)}')
log = log.replace('final_status=pending final zip verification', f'final_status={"PASS" if crit == 0 else "FAIL"}')
(REPORT / '10x_hotfix_execution_log.txt').write_text(log, encoding='utf-8-sig')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for src, rel in zip_items:
        zf.write(src, rel)

print({'hotfix_final_fail_count': crit, 'duplicate_entries': len(dup_names), 'zip_item_count': len(zip_items), 'zip_path': str(ZIP_PATH)})

{'hotfix_final_fail_count': 0, 'duplicate_entries': 0, 'zip_item_count': 33, 'zip_path': 'C:\\Code\\ott-churn-prediction\\park.ingyeom\\zip\\10x_feature_distribution_redundancy_pre_audit_260516_hotfix_review_package.zip'}